# MLB Capstone
## Wins and Payroll Merge

In [1]:
import pandas as pd
import numpy as np

In [2]:
payroll = pd.read_csv("../data/mlb_payroll_clean_2000_2025.csv")

In [3]:
payroll.head()

,team,year,payroll
0,Arizona Diamondbacks,2000,81027833
1,Atlanta Braves,2000,84537836
2,Baltimore Orioles,2000,81447435
3,Boston Red Sox,2000,77940333
4,Chicago Cubs,2000,60539333


In [4]:
wins_path = r"C:\Users\tlles\Documents\DA15\Capstone\baseball_payroll_wins_capstone\data\wins_csv.csv"

wins_wide = pd.read_csv(wins_path)
wins_wide.columns = [c.strip() for c in wins_wide.columns]

In [5]:
non_team_cols = {"Year", "G"}
team_cols = [c for c in wins_wide.columns if c not in non_team_cols]

In [6]:
wins_long = wins_wide.melt(
    id_vars=["Year","G"],
    value_vars=team_cols,
    var_name="team_abbr",
    value_name="wins"
)

In [7]:
wins_long.head()

,Year,G,team_abbr,wins
0,2025,162,ARI,80
1,2024,162,ARI,89
2,2023,162,ARI,84
3,2022,162,ARI,74
4,2021,162,ARI,52


In [8]:
wins_long["year"] = pd.to_numeric(wins_long["Year"], errors="coerce").astype("Int64")
wins_long["games"] = pd.to_numeric(wins_long["G"], errors="coerce").astype("Int64")
wins_long["wins"] = pd.to_numeric(wins_long["wins"], errors="coerce").astype("Int64")

wins_long = wins_long.drop(columns=["Year","G"])

In [9]:
wins_long.head()

,team_abbr,wins,year,games
0,ARI,80,2025,162
1,ARI,89,2024,162
2,ARI,84,2023,162
3,ARI,74,2022,162
4,ARI,52,2021,162


In [10]:
ABBR_TO_TEAM = {
    # AL West 
    "ATH": "Oakland Athletics",      
    "LAA": "Los Angeles Angels",
    "SEA": "Seattle Mariners",
    "TEX": "Texas Rangers",
    "HOU": "Houston Astros",

    # AL East
    "BAL": "Baltimore Orioles",
    "BOS": "Boston Red Sox",
    "NYY": "New York Yankees",
    "TBR": "Tampa Bay Rays",
    "TOR": "Toronto Blue Jays",

    # AL Central
    "CHW": "Chicago White Sox",
    "CLE": "Cleveland Guardians",    
    "DET": "Detroit Tigers",
    "KCR": "Kansas City Royals",
    "MIN": "Minnesota Twins",

    # NL West
    "ARI": "Arizona Diamondbacks",
    "COL": "Colorado Rockies",
    "LAD": "Los Angeles Dodgers",
    "SDP": "San Diego Padres",
    "SFG": "San Francisco Giants",

    # NL East
    "ATL": "Atlanta Braves",
    "MIA": "Miami Marlins",
    "NYM": "New York Mets",
    "PHI": "Philadelphia Phillies",
    "WSN": "Washington Nationals",   # Nationals (post-2005)
    

    # NL Central
    "CHC": "Chicago Cubs",
    "CIN": "Cincinnati Reds",
    "MIL": "Milwaukee Brewers",
    "PIT": "Pittsburgh Pirates",
    "STL": "St. Louis Cardinals",
}

wins_long["team"] = wins_long["team_abbr"].map(ABBR_TO_TEAM)

In [11]:
wins_long.head(6)

,team_abbr,wins,year,games,team
0,ARI,80,2025,162,Arizona Diamondbacks
1,ARI,89,2024,162,Arizona Diamondbacks
2,ARI,84,2023,162,Arizona Diamondbacks
3,ARI,74,2022,162,Arizona Diamondbacks
4,ARI,52,2021,162,Arizona Diamondbacks
5,ARI,25,2020,60,Arizona Diamondbacks


In [12]:
wins_long["win_pct"] = (wins_long["wins"] / wins_long["games"]).astype(float)

In [13]:
wins_long.head()

,team_abbr,wins,year,games,team,win_pct
0,ARI,80,2025,162,Arizona Diamondbacks,0.493827
1,ARI,89,2024,162,Arizona Diamondbacks,0.549383
2,ARI,84,2023,162,Arizona Diamondbacks,0.518519
3,ARI,74,2022,162,Arizona Diamondbacks,0.456790
4,ARI,52,2021,162,Arizona Diamondbacks,0.320988


In [14]:
merged = (wins_long
          .merge(payroll, on=["team","year"], how="inner")
          .sort_values(["year","team"])
          .reset_index(drop=True))

print("Rows:", len(merged))
print("Years:", merged["year"].min(), "–", merged["year"].max())
print("Per-year rows:\n", merged.groupby("year").size())

Rows: 780
Years: 2000 – 2025
Per-year rows:
 year
2000    30
2001    30
2002    30
2003    30
2004    30
2005    30
2006    30
2007    30
2008    30
2009    30
2010    30
2011    30
2012    30
2013    30
2014    30
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
2025    30
dtype: int64


In [15]:
merged.head()

,team_abbr,wins,year,games,team,win_pct,payroll
0,ARI,85,2000,162,Arizona Diamondbacks,0.524691,81027833
1,ATL,95,2000,162,Atlanta Braves,0.586420,84537836
2,BAL,74,2000,162,Baltimore Orioles,0.456790,81447435
3,BOS,85,2000,162,Boston Red Sox,0.524691,77940333
4,CHC,65,2000,162,Chicago Cubs,0.401235,60539333


In [21]:
merged.to_csv("mlb_payroll_wins_merge.csv", index=False)